# Claude Skills Tutorial

This notebook demonstrates how to use Claude Skills — organized packages of instructions,
executable code, and resources that give Claude specialized capabilities for specific tasks.

## Prerequisites
- Python 3.8+
- An Anthropic API key from [console.anthropic.com](https://console.anthropic.com)
- Virtual environment activated with dependencies installed

## 1. Setup & Installation Check

Run the cell below to verify your environment is set up correctly.

In [ ]:
import sys
import importlib

print(f"Python version: {sys.version}")
print()

required = ["anthropic", "dotenv"]
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "installed")
        print(f"  {pkg}: {version}")
    except ImportError:
        print(f"  {pkg}: NOT INSTALLED")

# Check anthropic version
import anthropic
from packaging.version import Version
if Version(anthropic.__version__) < Version("0.71.0"):
    print(f"\nanthropic SDK version {anthropic.__version__} is too old.")
    print("Run: pip install anthropic>=0.71.0")
else:
    print(f"\nAll checks passed!")

## 2. API Configuration

Load the API key and configure the client.

**Important:** Create a `.env` file in the project root directory:
```bash
cp ../.env.example ../.env
```
Then edit `../.env` to add your Anthropic API key.

In [ ]:
import os
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

from anthropic import Anthropic
from dotenv import load_dotenv

# Import our file utilities
from file_utils import (
    download_all_files,
    extract_file_ids,
    get_file_info,
    print_download_summary,
)

# Load environment variables from parent directory
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# Initialize client
client = Anthropic(api_key=API_KEY)

# Create outputs directory if it doesn't exist
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("API key loaded")
print(f"Using model: {MODEL}")
print(f"Output directory: {OUTPUT_DIR}")
print("\nNote: Beta headers will be added per-request when using Skills")

## 3. Test Connection

Verify the API connection works.

In [ ]:
# Simple test to verify API connection
test_response = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say 'Connection successful!' if you can read this.",
        }
    ],
)

print("API Test Response:")
print(test_response.content[0].text)
print(
    f"\nToken usage: {test_response.usage.input_tokens} in, {test_response.usage.output_tokens} out"
)

## 4. Discovering Available Skills

Let's discover what Anthropic-managed skills are available.

In [ ]:
# List all available Anthropic skills
client_with_skills_beta = Anthropic(
    api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"}
)

skills_response = client_with_skills_beta.beta.skills.list(source="anthropic")

print("Available Anthropic-Managed Skills:")
print("=" * 80)

for skill in skills_response.data:
    print(f"\nSkill ID: {skill.id}")
    print(f"   Title: {skill.display_title}")
    print(f"   Latest Version: {skill.latest_version}")
    print(f"   Created: {skill.created_at}")

    try:
        version_info = client_with_skills_beta.beta.skills.versions.retrieve(
            skill_id=skill.id, version=skill.latest_version
        )
        print(f"   Name: {version_info.name}")
        print(f"   Description: {version_info.description}")
    except Exception as e:
        print(f"   (Unable to fetch version details: {e})")

print(f"\n\nFound {len(skills_response.data)} Anthropic-managed skills")

## 5. Excel: Monthly Budget Spreadsheet

Create a budget spreadsheet with income/expenses, formulas, and a chart.

**Note:** Excel generation typically takes 1-2 minutes. Be patient!

In [ ]:
# Create an Excel budget spreadsheet
excel_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "xlsx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a monthly budget Excel spreadsheet with the following:

Income:
- Salary: $5,000
- Freelance: $1,200
- Investments: $300

Expenses:
- Rent: $1,500
- Utilities: $200
- Groceries: $600
- Transportation: $300
- Entertainment: $400
- Savings: $1,000

Include:
1. Formulas to calculate total income and total expenses
2. A formula for net savings (income - expenses)
3. Format currency values properly
4. Add a simple column chart showing income vs expenses
5. Use professional formatting with headers""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("Excel Response:")
print("=" * 80)
for content in excel_response.content:
    if content.type == "text":
        print(content.text)
    elif content.type == "tool_use":
        print(f"\nTool: {content.name}")
        if hasattr(content, "input"):
            print(f"   Input preview: {str(content.input)[:200]}...")

print(f"\n\nToken Usage:")
print(f"   Input: {excel_response.usage.input_tokens}")
print(f"   Output: {excel_response.usage.output_tokens}")

In [ ]:
# Download the Excel file
file_ids = extract_file_ids(excel_response)

if file_ids:
    print(f"Found {len(file_ids)} file(s)\n")

    results = download_all_files(
        client, excel_response, output_dir=str(OUTPUT_DIR), prefix="budget_"
    )

    print_download_summary(results)

    for file_id in file_ids:
        info = get_file_info(client, file_id)
        if info:
            print("\nFile Details:")
            print(f"   Filename: {info['filename']}")
            print(f"   Size: {info['size'] / 1024:.1f} KB")
            print(f"   Created: {info['created_at']}")
else:
    print("No files found in response")
    print("\nDebug: Response content types:")
    for i, content in enumerate(excel_response.content):
        print(f"  {i}. {content.type}")

## 6. PowerPoint: Revenue Presentation

Create a simple 2-slide presentation with a chart.

**Note:** PowerPoint generation typically takes 1-2 minutes.

In [ ]:
# Create a PowerPoint presentation
pptx_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pptx", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple 2-slide PowerPoint presentation:

Slide 1: Title slide
- Title: "Q3 2025 Results"
- Subtitle: "Acme Corporation"

Slide 2: Revenue Overview
- Title: "Quarterly Revenue"
- Add a simple column chart showing:
  - Q1: $12M
  - Q2: $13M
  - Q3: $14M

Use clean, professional formatting.""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PowerPoint Response:")
print("=" * 80)
for content in pptx_response.content:
    if content.type == "text":
        print(content.text)

print(f"\n\nToken Usage:")
print(f"   Input: {pptx_response.usage.input_tokens}")
print(f"   Output: {pptx_response.usage.output_tokens}")

In [ ]:
# Download the PowerPoint file
file_ids = extract_file_ids(pptx_response)

if file_ids:
    results = download_all_files(
        client, pptx_response, output_dir=str(OUTPUT_DIR), prefix="q3_review_"
    )

    print_download_summary(results)
    print("\nOpen the presentation in PowerPoint or Google Slides to view!")
else:
    print("No files found in response")

## 7. PDF: Simple Receipt

Create a receipt PDF with clean formatting.

**Note:** PDF generation typically takes 40-60 seconds.

In [ ]:
# Create a PDF receipt
pdf_response = client.beta.messages.create(
    model=MODEL,
    max_tokens=4096,
    container={"skills": [{"type": "anthropic", "skill_id": "pdf", "version": "latest"}]},
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    messages=[
        {
            "role": "user",
            "content": """Create a simple receipt PDF:

RECEIPT

Acme Corporation
Date: January 15, 2025
Receipt #: RCT-2025-001

Customer: Jane Smith

Items:
- Product A: $50.00
- Product B: $75.00
- Product C: $25.00

Subtotal: $150.00
Tax (8%): $12.00
Total: $162.00

Thank you for your business!

Use simple, clean formatting with clear sections.""",
        }
    ],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14", "skills-2025-10-02"],
)

print("PDF Response:")
print("=" * 80)
for content in pdf_response.content:
    if content.type == "text":
        print(content.text)

print(f"\n\nToken Usage:")
print(f"   Input: {pdf_response.usage.input_tokens}")
print(f"   Output: {pdf_response.usage.output_tokens}")

In [ ]:
# Download and verify the PDF
file_ids = extract_file_ids(pdf_response)

if file_ids:
    results = download_all_files(
        client, pdf_response, output_dir=str(OUTPUT_DIR), prefix="receipt_"
    )

    print_download_summary(results)

    # Verify PDF integrity
    for result in results:
        if result["success"]:
            file_path = result["output_path"]
            file_size = result["size"]

            with open(file_path, "rb") as f:
                header = f.read(5)
                if header == b"%PDF-":
                    print(f"\nPDF file is valid: {file_path}")
                    print(f"   File size: {file_size / 1024:.1f} KB")
                else:
                    print(f"\nFile may not be a valid PDF: {file_path}")
else:
    print("No files found in response")

## 8. Troubleshooting

### Common Issues

| Issue | Solution |
|-------|----------|
| `ANTHROPIC_API_KEY not found` | Create `.env` in project root with your key |
| `unexpected keyword argument 'container'` | Use `client.beta.messages.create()` |
| `Skills beta requires code_execution tool` | Include `code_execution` tool in request |
| `No files found in response` | Check code execution tool is included |
| `File not found` | Download files immediately after creation |

### Key Points
- Always use `client.beta.messages.create()` for Skills (not `client.messages.create()`)
- Always include the `code_execution` tool when using Skills
- Include all three betas: `code-execution-2025-08-25`, `files-api-2025-04-14`, `skills-2025-10-02`
- Document generation takes 40s-2min depending on complexity